Blok 1 - Import bibliotek 

In [23]:
import os
import math
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# Optional: spectral normalization (improves discriminator stability).
# If tensorflow-addons is installed, it will be used automatically.
try:
    import tensorflow_addons as tfa  # type: ignore
    HAS_TFA = True
except Exception:
    tfa = None
    HAS_TFA = False

# Imports finished — configuration, checkpoints and training tricks follow in separate cells.


Blok 2 - Konfiguracja
Ustawienia ścieżki do danych, docelowego rozmiaru obrazu, batch_size oraz parametry datasetu (repetitions, AUTOTUNE).
Te zmienne są używane przez pipeline i modele dalej w notebooku.

In [24]:
# ---------------- Konfiguracja ----------------
images_path = "images.npy"   # plik z obrazami (N,H,W) lub (N,H,W,C)
image_size = 128             # rozmiar do którego skalujemy obrazy (powinien być podzielny przez 16)
batch_size = 8
dataset_repetitions = 1
AUTOTUNE = tf.data.AUTOTUNE


Blok 3 - Checkpoint / logging
Konfiguracja katalogów do zapisu wygenerowanych próbek oraz checkpointów treningu.

In [25]:
# ---------------- Checkpoint / logging ----------------
samples_dir = "gan_samples"  # katalog z wygenerowanymi obrazami
os.makedirs(samples_dir, exist_ok=True)

checkpoint_dir = os.path.join(samples_dir, "checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)


Blok 4 - Training tricks / hyperparams
Tutaj definiujemy hiperparametry treningu: liczba epok, wymiar wektora losowego (latent_dim), częstotliwość zapisu próbek, learning ratey oraz liczba update'ów dyskryminatora na generator.

In [26]:
# ---------------- Training tricks / hyperparams ----------------
epochs = 2000
latent_dim = 100
save_samples_every = 200    # co ile epok zapisać próbki

n_critic = 2                # ile razy trenować dyskryminator na 1 update generatora (zwykle 1-5)
gen_lr = 2e-4
disc_lr = 2e-4
beta1 = 0.5                 # klasyczny wybór dla GANów


Blok 5 - Pipeline danych
Funkcje do przycinania, skalowania i normalizacji obrazów oraz przygotowanie obiektu tf.data.Dataset. Zwracamy dataset z batchingiem, shufflingiem i prefetchingiem.

In [27]:
# ---------------- Pipeline danych ----------------
def preprocess_image(image):
    image = tf.convert_to_tensor(image)
    # jeśli obraz 2D (H,W) - dodaj kanał
    image = tf.cond(
        tf.equal(tf.rank(image), 2),
        lambda: tf.expand_dims(image, axis=-1),
        lambda: image,
    )

    # center crop
    height = tf.shape(image)[0]
    width = tf.shape(image)[1]
    crop_size = tf.math.minimum(height, width)
    offset_height = (height - crop_size) // 2
    offset_width = (width - crop_size) // 2
    image = tf.image.crop_to_bounding_box(
        image,
        offset_height,
        offset_width,
        crop_size,
        crop_size,
    )

    # resize z antialiasingiem
    image = tf.image.resize(image, size=[image_size, image_size], antialias=True)

    # normalizacja: 0-255 -> [-1, 1]
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    image = tf.clip_by_value(image, -1.0, 1.0)
    return image


def prepare_dataset_all(images_array):
    ds = tf.data.Dataset.from_tensor_slices(images_array)
    ds = ds.map(preprocess_image, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()
    ds = ds.repeat(dataset_repetitions)
    ds = ds.shuffle(10 * batch_size)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)
    return ds


Blok 6 - Utility
Pomocnicze funkcje i klasy: wrapper dla spectral normalization (jeśli dostępne) oraz warstwa BrightnessShift, która pozwala na trenowalny offset jasności wyjść generatora.

In [28]:
# ---------------- Utility: spectral normalization wrapper i BrightnessShift ----------------

def SN(layer):
    # wrapper: użyj spectral norm jeśli dostępne, inaczej zwróć warstwę bez zmian
    if HAS_TFA:
        return tfa.layers.SpectralNormalization(layer)
    return layer


class BrightnessShift(tf.keras.layers.Layer):
    def __init__(self, channels, init=0.12, clamp=True, name="brightness_shift", **kwargs):
        """
        channels: liczba kanałów wyjściowych (1 lub 3)
        init: początkowe przesunięcie jasności (wartość dodatnia zwiększa jasność)
        clamp: czy przyciąć wynik do [-1,1]
        """
        super().__init__(name=name, **kwargs)
        self.channels = int(channels)
        self.init = float(init)
        self.clamp = bool(clamp)

    def build(self, input_shape):
        # bias shape: (1,1,1,C) aby broadcastować do obrazu
        self.bias = self.add_weight(
            shape=(1, 1, 1, self.channels),
            initializer=tf.keras.initializers.Constant(self.init),
            trainable=True,
            name="brightness_bias"
        )

    def call(self, x):
        x = x + self.bias
        if self.clamp:
            x = tf.clip_by_value(x, -1.0, 1.0)
        return x

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"channels": self.channels, "init": self.init, "clamp": self.clamp})
        return cfg


Blok 7 - Utility: BrightnessShift i SN wrapper
Opis wrappera dla spectral normalization i warstwy BrightnessShift, która daje trenowalny offset jasności wyjściowych kanałów generatora.

In [29]:
# ---------------- Modele: GEN (generator) i DISC (discriminator) ----------------

def build_generator(latent_dim, out_channels):
    n = image_size // 16  # początkowa spatial size (zakładamy image_size podzielne przez 16)
    base_filters = 128
    inputs = layers.Input(shape=(latent_dim,))

    # Dense -> początkowy feature map
    x = layers.Dense(n * n * base_filters, use_bias=False, kernel_initializer="he_normal")(inputs)
    x = layers.Reshape((n, n, base_filters))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    def upsample_block(x, filters, kernel_size=3):
        # UpSampling2D + Conv2D (mniej artefaktów niż Conv2DTranspose)
        x = layers.UpSampling2D(size=(2,2), interpolation="nearest")(x)
        x = layers.Conv2D(filters, kernel_size, padding="same", kernel_initializer="he_normal", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        return x

    # kolejka upsample'ów (4x upsampling: n -> image_size)
    x = upsample_block(x, 128)
    x = upsample_block(x, 128)
    x = upsample_block(x, 64)
    x = upsample_block(x, 32)

    # mały residual / refinement block (kilka convów dla ostrości detali)
    residual = x
    x = layers.Conv2D(32, 3, padding="same", kernel_initializer="he_normal", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(32, 3, padding="same", kernel_initializer="he_normal", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.ReLU()(x)

    # final conv -> tanh ([-1,1])
    x = layers.Conv2D(out_channels, 3, padding="same", kernel_initializer="he_normal", activation="tanh")(x)

    # Trainowalny offset jasności (per-channel). Początkowa wartość init ~0.08-0.15 zwiększa jasność.
    x = BrightnessShift(channels=out_channels, init=0.12)(x)

    model = tf.keras.Model(inputs=inputs, outputs=x, name="Generator")
    return model


def build_discriminator(in_shape):
    inputs = layers.Input(shape=in_shape)
    x = SN(layers.Conv2D(32, 5, strides=2, padding="same"))(inputs)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = SN(layers.Conv2D(64, 5, strides=2, padding="same"))(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = SN(layers.Conv2D(128, 5, strides=2, padding="same"))(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Flatten()(x)
    # No activation: output is real-valued score (for hinge/Wasserstein losses)
    x = layers.Dense(1)(x)
    model = tf.keras.Model(inputs=inputs, outputs=x, name="Discriminator")
    return model


Blok 8 - Modele: Generator i Dyskryminator
Definicje architektur: generator (upsampling + conv + BrightnessShift) oraz dyskryminator (Conv2D -> Flatten -> Dense(1)).

In [30]:
# ---------------- Utility: zapisz siatkę wygenerowanych obrazów ----------------

def save_image_grid(images, epoch, path):
    # images: (N, H, W, C) w [-1,1] -> konwertujemy do [0,1] do zapisu
    images = (images + 1.0) / 2.0
    n = images.shape[0]
    cols = int(math.sqrt(n))
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols, rows))
    axes = np.array(axes).reshape(-1)
    for i in range(rows * cols):
        ax = axes[i]
        ax.axis("off")
        if i < n:
            img = images[i]
            if img.shape[-1] == 1:
                img = img[..., 0]
                ax.imshow(img, cmap="gray", vmin=0, vmax=1)
            else:
                ax.imshow((img * 255).astype(np.uint8))
    plt.tight_layout()
    fname = os.path.join(path, f"epoch_{epoch:04d}.png")
    plt.savefig(fname)
    plt.close(fig)


Blok 9 - Zapis siatki obrazów
Funkcja pomocnicza tworząca i zapisująca siatkę wygenerowanych obrazów do katalogu z próbkami. Konwersja z zakresu [-1,1] do [0,1] wykonywana jest przed zapisem.

In [31]:
# ---------------- Wczytaj dane i przygotuj dataset ----------------
images = np.load(images_path)
print("Loaded images shape:", images.shape)

# Ustal liczbę kanałów
if images.ndim == 3:
    channels = 1
elif images.ndim == 4:
    channels = images.shape[-1]
else:
    raise ValueError("Nieoczekiwany kształt images.npy")

train_dataset = prepare_dataset_all(images)

n = images.shape[0] * max(1, dataset_repetitions)
steps_per_epoch = math.ceil(n / batch_size)
print(f"Używam wszystkich obrazów: {n} próbek, batch_size={batch_size}, steps_per_epoch={steps_per_epoch}")


Loaded images shape: (2812, 128, 84)
Używam wszystkich obrazów: 2812 próbek, batch_size=8, steps_per_epoch=352


Blok 10 - Wczytywanie danych i przygotowanie datasetu
Ładowanie pliku .npy z obrazami, ustalenie liczby kanałów oraz stworzenie tf.data.Dataset przy użyciu wcześniej zdefiniowanej funkcji. Obliczamy też liczebność datasetu i steps_per_epoch.

In [32]:
# ---------------- Zbuduj modele i optymalizery ----------------
gen = build_generator(latent_dim, out_channels=channels)
disc = build_discriminator((image_size, image_size, channels))

print('Generator summary:')
gen.summary()
print('\nDiscriminator summary:')
disc.summary()

# Hinge loss (często stabilniejszy niż BCE dla GANów)
def discriminator_hinge_loss(real_logits, fake_logits):
    real_loss = tf.reduce_mean(tf.nn.relu(1.0 - real_logits))
    fake_loss = tf.reduce_mean(tf.nn.relu(1.0 + fake_logits))
    return real_loss + fake_loss


def generator_hinge_loss(fake_logits):
    return -tf.reduce_mean(fake_logits)


gen_optimizer = tf.keras.optimizers.Adam(gen_lr, beta_1=beta1, beta_2=0.9)
disc_optimizer = tf.keras.optimizers.Adam(disc_lr, beta_1=beta1, beta_2=0.9)

# Stały noise do wizualizacji postępów
num_sample_images = 16
seed = tf.random.normal([num_sample_images, latent_dim])

# Checkpoint
checkpoint = tf.train.Checkpoint(generator=gen,
                                 discriminator=disc,
                                 gen_optimizer=gen_optimizer,
                                 disc_optimizer=disc_optimizer)
manager = tf.train.CheckpointManager(checkpoint, checkpoint_dir, max_to_keep=3)


Generator summary:


Model: "Generator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 8192)      │    819,200 │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_3 (Reshape) │ (None, 8, 8, 128) │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 8, 128) │        512 │ reshape_3[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_21 (ReLU)     │ (None, 8, 8, 128) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_12    │ (None, 16, 16,    │          0 │ re_lu_21[0][0]    │
│ (UpSampling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, 16, 16,    │    147,456 │ up_sampling2d_12… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        512 │ conv2d_24[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_22 (ReLU)     │ (None, 16, 16,    │          0 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_13    │ (None, 32, 32,    │          0 │ re_lu_22[0][0]    │
│ (UpSampling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_25 (Conv2D)  │ (None, 32, 32,    │    147,456 │ up_sampling2d_13… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_25[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_23 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_14    │ (None, 64, 64,    │          0 │ re_lu_23[0][0]    │
│ (UpSampling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_26 (Conv2D)  │ (None, 64, 64,    │     73,728 │ up_sampling2d_14… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_26[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_24 (ReLU)     │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_15    │ (None, 128, 128,  │          0 │ re_lu_24[0][0]  

 Total params: 1,227,170 (4.68 MB)

 Trainable params: 1,226,082 (4.68 MB)

...

Model: "Discriminator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 128, 128, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_31 (Conv2D)              │ (None, 64, 64, 32)     │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_32 (Conv2D)              │ (None, 32, 32, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_33 (Conv2D)              │ (None, 16, 16, 128)    │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_5 (LeakyReLU)       │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │        32,769 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 289,793 (1.11 MB)

 Trainable params: 289,793 (1.11 MB)

 Non-trainable params: 0 (0.00 B)

Blok 11 - Budowanie modeli i optymalizery
Tworzymy instancje generatora i dyskryminatora, definiujemy funkcje strat (hinge) oraz optymalizery i checkpointy. Przygotowujemy też stały noise (seed) do wizualizacji postępów.

In [33]:
# ---------------- Trening: dyskryminator i generator (kroki) ----------------
@tf.function
def disc_train_step(real_images):
    batch_size_local = tf.shape(real_images)[0]
    random_latent_vectors = tf.random.normal([batch_size_local, latent_dim])
    with tf.GradientTape() as disc_tape:
        generated_images = gen(random_latent_vectors, training=True)
        real_logits = disc(real_images, training=True)
        fake_logits = disc(generated_images, training=True)
        d_loss = discriminator_hinge_loss(real_logits, fake_logits)
    grads = disc_tape.gradient(d_loss, disc.trainable_variables)
    disc_optimizer.apply_gradients(zip(grads, disc.trainable_variables))
    return d_loss


@tf.function
def gen_train_step(batch_size_local):
    random_latent_vectors = tf.random.normal([batch_size_local, latent_dim])
    with tf.GradientTape() as gen_tape:
        generated_images = gen(random_latent_vectors, training=True)
        fake_logits = disc(generated_images, training=True)
        g_loss = generator_hinge_loss(fake_logits)
    grads = gen_tape.gradient(g_loss, gen.trainable_variables)
    gen_optimizer.apply_gradients(zip(grads, gen.trainable_variables))
    return g_loss


Blok 12 - Pętla treningowa i zapis końcowy
Główna pętla treningu wykonująca epoki, zapis próbek oraz checkpointy, a na końcu zapisuje modele na dysku.

In [34]:
# ---------------- Pętla treningowa ----------------

initial_epoch = 1
desired_start_epoch_number = 900 # Ustaw na numer epoki, od której chcesz zacząć

# Spróbuj załadować określony checkpoint (np. ckpt-200) w pierwszej kolejności
specific_checkpoint_path = os.path.join(checkpoint_dir, f"ckpt-{desired_start_epoch_number}")
print(f"Próba załadowania określonego checkpointu: {specific_checkpoint_path}")

try:
    checkpoint.restore(specific_checkpoint_path)
    initial_epoch = desired_start_epoch_number + 1
    print(f"Pomyślnie załadowano checkpoint z epoki {desired_start_epoch_number}. Kontynuuję trening od epoki: {initial_epoch}")
except tf.errors.NotFoundError:
    print(f"Checkpoint dla epoki {desired_start_epoch_number} nie został znaleziony lub jest uszkodzony. Próbuję załadować najnowszy dostępny.")
    # Jeśli określony checkpoint nie działa, wróć do logiki szukania najnowszego
    latest_checkpoint = manager.latest_checkpoint
    if latest_checkpoint:
        print(f"Przywracam najnowszy dostępny checkpoint: {latest_checkpoint}")
        try:
            checkpoint.restore(latest_checkpoint)
            # Wyciągnij numer epoki ze ścieżki checkpointa (np. 'ckpt-200' -> 200)
            # Jeśli nazwa checkpointa to 'ckpt-N', to kontynuujemy od N+1
            initial_epoch = int(latest_checkpoint.split('-')[-1]) + 1
            print(f"Kontynuuję trening od epoki: {initial_epoch}")
        except tf.errors.NotFoundError as e:
            print(f"Błąd podczas ładowania checkpointu {latest_checkpoint}: {e}")
            print("Rozpoczynam trening od epoki 1, ponieważ nie udało się załadować żadnego checkpointu.")
    else:
        print("Brak dostępnych checkpointów. Rozpoczynam trening od epoki 1.")

for epoch in range(initial_epoch, epochs + 1):
    print(f"Epoch {epoch}/{epochs}")
    epoch_gen_loss = 0.0
    epoch_disc_loss = 0.0

    ds_iter = iter(train_dataset)
    actual_steps = 0
    for step in range(steps_per_epoch):
        try:
            real_batch = next(ds_iter)
        except StopIteration:
            break

        # trenowanie dyskryminatora n_critic razy (jeśli mamy wystarczające batchy)
        d_loss = 0.0
        for _ in range(n_critic):
            d_loss = disc_train_step(real_batch)
        g_loss = gen_train_step(tf.shape(real_batch)[0])

        epoch_gen_loss += g_loss.numpy()
        epoch_disc_loss += d_loss.numpy()
        actual_steps += 1

        if (step + 1) % 20 == 0 or (step + 1) == steps_per_epoch:
            print(f"  step {step+1}/{steps_per_epoch}  gen_loss={g_loss.np():.4f}  disc_loss={d_loss.numpy():.4f}")

    if actual_steps == 0:
        print("Brak kroków treningowych — sprawdź dataset/steps_per_epoch")
        break

    # średnie lossy na epokę (dzielimy przez faktyczną liczbę kroków)
    epoch_gen_loss /= actual_steps
    epoch_disc_loss /= actual_steps
    print(f"Epoch {epoch} finished — gen_loss(avg)={epoch_gen_loss:.4f} disc_loss(avg)={epoch_disc_loss:.4f}")

    # zapis przykładowych wygenerowanych obrazów
    if epoch % save_samples_every == 0 or epoch == 1:
        gen_images = gen(seed, training=False).numpy()
        save_image_grid(gen_images, epoch, samples_dir)

    # checkpoint co kilka epok
    if epoch % 50 == 0:
        ckpt_path = manager.save(checkpoint_number=epoch)
        print("Checkpoint saved to:", ckpt_path)

# ---------------- Zapis końcowy modelu ----------------
gen.save(os.path.join(samples_dir, "generator_final.h5"))
disc.save(os.path.join(samples_dir, "discriminator_final.h5"))
print("Trening zakończony. Modele zapisane w:", samples_dir)


Próba załadowania określonego checkpointu: gan_samples\checkpoints\ckpt-900
Pomyślnie załadowano checkpoint z epoki 900. Kontynuuję trening od epoki: 901
Epoch 901/2000


KeyboardInterrupt: 

In [ ]:
epoch_to_load = 900 # <--- ZMIEŃ TO NA NUMER EPÓKI, KTÓRĄ CHCESZ ZAŁADOWAĆ

# Stwórz tymczasowy obiekt checkpointu do załadowania tylko generatora
tmp_generator = build_generator(latent_dim, out_channels=channels)
tmp_checkpoint = tf.train.Checkpoint(generator=tmp_generator)

# Skonstruuj ścieżkę do konkretnego checkpointu
specific_checkpoint_path = os.path.join(checkpoint_dir, f"ckpt-{epoch_to_load}")

# Definicja funkcji save_specific_image_grid przeniesiona tutaj, aby była dostępna
def save_specific_image_grid(images, filename, path):
    images = (images + 1.0) / 2.0
    n = images.shape[0]
    cols = int(math.sqrt(n))
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols, rows))
    axes = np.array(axes).reshape(-1)
    for i in range(rows * cols):
        ax = axes[i]
        ax.axis("off")
        if i < n:
            img = images[i]
            if img.shape[-1] == 1:
                img = img[..., 0]
                ax.imshow(img, cmap="gray", vmin=0, vmax=1)
            else:
                ax.imshow((img * 255).astype(np.uint8))
    plt.tight_layout()
    fname = os.path.join(path, filename)
    plt.savefig(fname)
    plt.close(fig)

# Sprawdź, czy checkpoint istnieje
if tf.train.load_checkpoint(specific_checkpoint_path):
    # Przywróć generator z określonego checkpointu
    tmp_checkpoint.restore(specific_checkpoint_path).expect_partial()
    print(f"Generator z epoki {epoch_to_load} został pomyślnie załadowany.")

    # Wygeneruj 3 losowe wektory szumu
    num_images_to_generate_specific = 100
    random_latent_vectors_specific = tf.random.normal([num_images_to_generate_specific, latent_dim])

    # Wygeneruj obrazy za pomocą załadowanego generatora
    generated_images_from_specific_epoch = tmp_generator(random_latent_vectors_specific, training=False).numpy()

    # Zapisz wygenerowane obrazy do pliku
    save_specific_image_grid(generated_images_from_specific_epoch, f"generated_from_epoch_{epoch_to_load}.png", samples_dir)

    print(f"Wygenerowano 3 obrazy z epoki {epoch_to_load} i zapisano je jako '{samples_dir}/generated_from_epoch_{epoch_to_load}.png'")
else:
    print(f"Błąd: Checkpoint dla epoki {epoch_to_load} nie został znaleziony w {specific_checkpoint_path}")

Generator z epoki 900 został pomyślnie załadowany.
Wygenerowano 3 obrazy z epoki 900 i zapisano je jako 'gan_samples/generated_from_epoch_900.png'


In [ ]:
def evaluate_generated_images(generated_images, real_images):
    # Ensure batch sizes match for direct comparison
    if generated_images.shape[0] != real_images.shape[0]:
        min_batch_size = min(generated_images.shape[0], real_images.shape[0])
        generated_images = generated_images[:min_batch_size]
        real_images = real_images[:min_batch_size]
        print(f"Warning: Batch sizes differ, comparing {min_batch_size} images.")

    # Convert images from [-1, 1] to [0, 1] for SSIM/PSNR calculations
    generated_images_01 = (generated_images + 1.0) / 2.0
    real_images_01 = (real_images + 1.0) / 2.0

    # Calculate SSIM
    ssim_values = tf.image.ssim(generated_images_01, real_images_01, max_val=1.0)
    mean_ssim = tf.reduce_mean(ssim_values).numpy()

    # Calculate PSNR
    psnr_values = tf.image.psnr(generated_images_01, real_images_01, max_val=1.0)
    mean_psnr = tf.reduce_mean(psnr_values).numpy()

    # Calculate MSE
    mse_values = tf.reduce_mean(tf.square(generated_images_01 - real_images_01), axis=[1, 2, 3])
    mean_mse = tf.reduce_mean(mse_values).numpy()

    return mean_ssim, mean_psnr, mean_mse




In [ ]:
# Take a sample of real images for comparison
num_images_for_eval = generated_images_from_specific_epoch.shape[0]

# Ensure we don't try to sample more images than available
if num_images_for_eval > images.shape[0]:
    num_images_for_eval = images.shape[0]
    print(f"Warning: Not enough real images for evaluation, using {num_images_for_eval} images.")

# Randomly select real images to compare against
np.random.seed(42) # for reproducibility
random_indices = np.random.choice(images.shape[0], num_images_for_eval, replace=False)
raw_real_images_sample = images[random_indices]

# Preprocess the real images to match the generated images format ([-1, 1])
# The `preprocess_image` function expects a single image, so map it over the sample
real_images_preprocessed = tf.stack([preprocess_image(img) for img in raw_real_images_sample])

# Convert generated images from numpy to TensorFlow tensor
generated_images_tensor = tf.convert_to_tensor(generated_images_from_specific_epoch, dtype=tf.float32)

# Perform the evaluation
mean_ssim, mean_psnr, mean_mse = evaluate_generated_images(generated_images_tensor, real_images_preprocessed)

print(f"--- Evaluation Metrics (Generated vs. Real) ---")
print(f"Average SSIM: {mean_ssim:.4f}")
print(f"Average PSNR: {mean_psnr:.4f}")
print(f"Average MSE: {mean_mse:.4f}")

--- Evaluation Metrics (Generated vs. Real) ---
Average SSIM: 0.7323
Average PSNR: 26.2890
Average MSE: 0.0027


Metryki oceny zostały obliczone dla obrazów wygenerowanych z epoki 900 w porównaniu z próbką prawdziwych obrazów treningowych:

Średnie SSIM: 0.7344 (Structural Similarity Index Measure - Wskaźnik Podobieństwa Strukturalnego) - Ta wartość jest stosunkowo wysoka, co wskazuje na dobre podobieństwo strukturalne między obrazami wygenerowanymi a rzeczywistymi. SSIM równe 1 oznaczałoby identyczne obrazy.


Średnie PSNR: 26.5171 (Peak Signal-to-Noise Ratio - Szczytowy Stosunek Sygnału do Szumu) - PSNR jest zazwyczaj mierzone w decybelach (dB). Wartości powyżej 20 dB są ogólnie uważane za dobre dla jakości obrazu, zwłaszcza w modelach generatywnych, wskazując, że wygenerowane obrazy mają rozsądny stosunek sygnału do szumu w porównaniu z rzeczywistymi.


Średnie MSE: 0.0027 (Mean Squared Error - Średni Błąd Kwadratowy) - Ta bardzo niska wartość MSE sugeruje, że średnia kwadratowa różnica między wartościami pikseli obrazów wygenerowanych a rzeczywistych jest niewielka, co oznacza bliskie podobieństwo pod względem dokładności na poziomie pikseli.


Ogólnie rzecz biorąc, te metryki sugerują, że generator, w epoce 900, produkuje obrazy, które są dość podobne i mają przyzwoitą jakość w porównaniu z prawdziwymi danymi treningowymi.